In [1]:
import pandas as pd
import numpy as np
import os
import h3

In [2]:
# Define the folder path and the column names
# '/Users/jul/Desktop/uni/Data Analytics/project/PROBE-202411'
# /Users/jul/Desktop/uni/Data Analytics/PROBE-202409
# '/Users/jul/Desktop/uni/Data Analytics/PROBE-202412'
folder_paths = [
    '/Users/jul/Desktop/uni/Data Analytics/project/PROBE-202411'
]
columns = [
    'VehicleID',
    'gpsvalid',
    'lat',
    'lon',
    'timestamp',
    'speed',
    'heading',
    'for_hire_light',
    'engine_acc'
]

In [3]:
# Initialize an empty list to store the dataframes
all_dfs = []

In [4]:

# Loop through all folders
for folder_path in folder_paths:
    # Loop through all files in the directory
    for filename in os.listdir(folder_path):
        # Check if the file is a CSV file
        if filename.endswith('.csv.out'):
            file_path = os.path.join(folder_path, filename)

            # Read the CSV file into a dataframe with the specified column names
            df = pd.read_csv(file_path, names=columns)

            # Append the dataframe to the list
            all_dfs.append(df)

# Concatenate all dataframes in the list into a single dataframe
combined_df = pd.concat(all_dfs, ignore_index=True)

In [5]:
combined_df

,VehicleID,gpsvalid,lat,lon,timestamp,speed,heading,for_hire_light,engine_acc
0,t7K8v5g8YiXmnbuvfVH4t4qZydQ,1,13.85360,100.55141,2024-11-25 23:54:56,0,63,1,0
1,ySQq59oNQ+kk5G1bx796QOjUQ7M,1,13.71034,100.60201,2024-11-25 23:55:42,36,59,1,1
2,pfmJE+AxHlNbR6Nxhn9t3sDGx7k,1,13.84241,100.57710,2024-11-25 23:56:00,76,29,1,1
3,mz/5Q2opqjHf4kn5WZHoWuBB7VE,1,13.83726,100.60883,2024-11-25 23:55:35,0,285,0,0
4,6dvZqj2Qoq3FvsgF1dDiK827RjM,1,13.82041,100.57500,2024-11-25 23:56:47,61,342,0,1
...,...,...,...,...,...,...,...,...,...
55629589,J5DxZvsj/r9b6Kex8b2OLREv7/o,1,7.87370,98.29954,2024-11-23 23:59:47,0,52,0,0
55629590,QAbyFmQazKLgfGUkfUAPX3HCEIo,1,13.83596,100.64203,2024-11-23 23:59:46,0,64,0,1
55629591,I5AeppcV2mKC7aNKFtyf7WX7QR4,1,7.83091,98.34943,2024-11-23 23:59:47,0,187,0,0
55629592,R1hDu+U6GKLvdQVVVkVp5X/ChdE,1,14.77243,101.52135,2024-11-23 23:59:47,62,195,0,1


In [6]:
cleaned_df = combined_df.dropna()


In [7]:
cleaned_taxi_df = cleaned_df[
    (combined_df['gpsvalid'] == 1) &
    (combined_df['engine_acc'] == 1) &
    (combined_df['lat'].between(-90, 90)) &
    (combined_df['lon'].between(-180, 180)) &
    (combined_df['speed'] >= 0)
].copy().reset_index(drop=True)

In [8]:
# Find the unique VehicleIDs of every vehicle that EVER reported for_hire_light = 1
# This is our most reliable definition of a taxi.
taxi_ids = cleaned_taxi_df[cleaned_taxi_df['for_hire_light'] == 1]['VehicleID'].unique()

In [9]:
# Now, filter the main dataframe to keep ALL records for these identified taxis
# This gives us their full journey, not just when their light was on.
taxis_df = cleaned_taxi_df[cleaned_taxi_df['VehicleID'].isin(taxi_ids)].copy()

In [10]:
# The 'timestamp' column is just text right now. We need to convert it.
taxis_df['timestamp'] = pd.to_datetime(taxis_df['timestamp'])

Sort Data

In [11]:
# Sort by the vehicle first, then by the time.
taxis_df.sort_values(by=['VehicleID', 'timestamp'], inplace=True)

In [12]:
# Reset the index after sorting for a clean DataFrame
taxis_df.reset_index(drop=True, inplace=True)

In [13]:
# Extract time-based features from the timestamp
taxis_df['hour'] = taxis_df['timestamp'].dt.hour
taxis_df['day_of_week'] = taxis_df['timestamp'].dt.dayofweek # Monday=0, Sunday=6
taxis_df['is_weekend'] = (taxis_df['day_of_week'] >= 5).astype(int)

In [14]:
# --- Advanced: Identify Trips ---
# A trip starts when the 'for_hire_light' changes from 1 (empty) to 0 (occupied).
# We can create a 'trip_id' for each taxi.

# First, detect the change from 1 to 0
taxis_df['trip_start'] = (taxis_df['for_hire_light'].shift(1) == 1) & (taxis_df['for_hire_light'] == 0)

In [15]:
# We also need to ensure it's the same vehicle
taxis_df['trip_start'] = taxis_df['trip_start'] & (taxis_df['VehicleID'].shift(1) == taxis_df['VehicleID'])

In [16]:
# Now create a unique ID for each trip using a cumulative sum
taxis_df['trip_id'] = taxis_df.groupby('VehicleID')['trip_start'].cumsum()

In [17]:
# Let's clean up the intermediate column
taxis_df.drop(columns=['trip_start'], inplace=True)

In [18]:
taxis_df

,VehicleID,gpsvalid,lat,lon,timestamp,speed,heading,for_hire_light,engine_acc,hour,day_of_week,is_weekend,trip_id
0,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64477,100.63777,2024-11-01 08:32:41,0,109,1,1,8,4,0,0
1,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64477,100.63777,2024-11-01 08:34:41,0,109,1,1,8,4,0,0
2,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64477,100.63781,2024-11-01 08:36:41,0,109,1,1,8,4,0,0
3,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64477,100.63781,2024-11-01 08:38:42,0,109,1,1,8,4,0,0
4,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64477,100.63781,2024-11-01 08:40:41,0,109,1,1,8,4,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
16141363,zzLYPcDONaA8lLF2aJYFKnoRDQ4,1,13.92919,100.49025,2024-11-30 21:32:47,42,45,1,1,21,5,1,0
16141364,zzLYPcDONaA8lLF2aJYFKnoRDQ4,1,13.93869,100.49862,2024-11-30 21:34:47,32,35,1,1,21,5,1,0
16141365,zzLYPcDONaA8lLF2aJYFKnoRDQ4,1,13.94617,100.49917,2024-11-30 21:36:47,37,315,1,1,21,5,1,0
16141366,zzLYPcDONaA8lLF2aJYFKnoRDQ4,1,13.94629,100.49662,2024-11-30 21:38:47,0,297,1,1,21,5,1,0


In [19]:
# cars with trip id
taxis_df.groupby('VehicleID')["trip_id"].nunique()

VehicleID
++qQzutWwL31NcUo8s0jiGZzzS0      1
+1indEOKr/ikPVrJQTVjw4FGxBE      1
+20prWr63K5svsMtTmdqLnmsTGE    294
+A3arvBS15eOvdHE+E06+Ng28+E    245
+BAgYWCbvz0z377ef3Yp687Pp+0    364
                              ... 
zt6r6X6cjBVDqWxKcbyXkn1Cydw    255
zvkvMBxj2VDpNjIfmifwwxl6nMo     57
zwNAI8pHuCgPOF5NMs2nVaXBG04    298
zyy0Hv5yMoClNPQlx1tR4NBssFI      1
zzLYPcDONaA8lLF2aJYFKnoRDQ4      1
Name: trip_id, Length: 2016, dtype: int64

In [20]:
#how many taxis extracted
taxis_df['VehicleID'].nunique()

2016

In [21]:
#how many trips extracted
taxis_df.groupby('VehicleID')['trip_id'].max().sum()

np.int64(419447)

Calculate Idle Time

In [22]:
# It creates the unique ID for each continuous idle period.
taxis_df['idle_start'] = (taxis_df['for_hire_light'].shift(1) == 0) & \
                         (taxis_df['for_hire_light'] == 1) & \
                         (taxis_df['VehicleID'].shift(1) == taxis_df['VehicleID'])
taxis_df['idle_period_id'] = taxis_df.groupby('VehicleID')['idle_start'].cumsum()
taxis_df.loc[taxis_df['trip_id'] == 0, 'idle_period_id'] = 0

In [23]:
# Create a temporary DataFrame with only the idle data points
idle_periods_df = taxis_df[taxis_df['for_hire_light'] == 1].copy()

# Group by each unique idle period to calculate its duration
idle_summary = idle_periods_df.groupby(['VehicleID', 'idle_period_id']).agg(
    start_time=('timestamp', 'min'),
    end_time=('timestamp', 'max')
).reset_index()

# Calculate the duration in minutes
idle_summary['idle_duration_minutes'] = (idle_summary['end_time'] - idle_summary['start_time']).dt.total_seconds() / 60 + 1

print("--- This is the information we will merge back ---")
display(idle_summary[['VehicleID', 'idle_period_id', 'idle_duration_minutes']].head())

--- This is the information we will merge back ---


,VehicleID,idle_period_id,idle_duration_minutes
0,++qQzutWwL31NcUo8s0jiGZzzS0,0,42429.666667
1,+1indEOKr/ikPVrJQTVjw4FGxBE,0,42957.700000
2,+20prWr63K5svsMtTmdqLnmsTGE,0,29.000000
3,+20prWr63K5svsMtTmdqLnmsTGE,2,22.000000
4,+20prWr63K5svsMtTmdqLnmsTGE,3,32.000000


In [24]:
# We use a 'left' merge to ensure we keep ALL original rows from taxis_df
# The merge will add the 'idle_duration_minutes' from the summary to the main table
# based on the matching VehicleID and idle_period_id.
taxis_df = pd.merge(
    taxis_df,
    idle_summary[['VehicleID', 'idle_period_id', 'idle_duration_minutes']],
    on=['VehicleID', 'idle_period_id'],
    how='left'
)

# After merging, the rows that were NOT idle (i.e., part of a trip) will have NaN
# for the idle duration. We should fill these with 0.
taxis_df['idle_duration_minutes'].fillna(0, inplace=True)

/var/folders/qb/yf4211gs4kv7hy5d5jp7q6bw0000gn/T/ipykernel_95457/1755066191.py:13: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  taxis_df['idle_duration_minutes'].fillna(0, inplace=True)


In [25]:
taxis_df.drop(columns=['idle_start'], inplace=True)

In [26]:
taxis_df

,VehicleID,gpsvalid,lat,lon,timestamp,speed,heading,for_hire_light,engine_acc,hour,day_of_week,is_weekend,trip_id,idle_period_id,idle_duration_minutes
0,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64477,100.63777,2024-11-01 08:32:41,0,109,1,1,8,4,0,0,0,42429.666667
1,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64477,100.63777,2024-11-01 08:34:41,0,109,1,1,8,4,0,0,0,42429.666667
2,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64477,100.63781,2024-11-01 08:36:41,0,109,1,1,8,4,0,0,0,42429.666667
3,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64477,100.63781,2024-11-01 08:38:42,0,109,1,1,8,4,0,0,0,42429.666667
4,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64477,100.63781,2024-11-01 08:40:41,0,109,1,1,8,4,0,0,0,42429.666667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16141363,zzLYPcDONaA8lLF2aJYFKnoRDQ4,1,13.92919,100.49025,2024-11-30 21:32:47,42,45,1,1,21,5,1,0,0,42573.866667
16141364,zzLYPcDONaA8lLF2aJYFKnoRDQ4,1,13.93869,100.49862,2024-11-30 21:34:47,32,35,1,1,21,5,1,0,0,42573.866667
16141365,zzLYPcDONaA8lLF2aJYFKnoRDQ4,1,13.94617,100.49917,2024-11-30 21:36:47,37,315,1,1,21,5,1,0,0,42573.866667
16141366,zzLYPcDONaA8lLF2aJYFKnoRDQ4,1,13.94629,100.49662,2024-11-30 21:38:47,0,297,1,1,21,5,1,0,0,42573.866667


Filter out irrelavant date and time

In [27]:
unique_years = taxis_df['timestamp'].dt.year.unique()
unique_years.sort()
unique_years

array([1970, 2024], dtype=int32)

In [28]:
# filter out years more than 2024 and less than 2015
year_series = taxis_df['timestamp'].dt.year

# Keep only the rows where the year is between 2016 and 2024 (inclusive)
# "later than 2015" means >= 2016
# "drop further than 2024" means <= 2024

taxis_df = taxis_df[(year_series == 2024)].copy()

In [29]:
taxis_df

,VehicleID,gpsvalid,lat,lon,timestamp,speed,heading,for_hire_light,engine_acc,hour,day_of_week,is_weekend,trip_id,idle_period_id,idle_duration_minutes
0,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64477,100.63777,2024-11-01 08:32:41,0,109,1,1,8,4,0,0,0,42429.666667
1,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64477,100.63777,2024-11-01 08:34:41,0,109,1,1,8,4,0,0,0,42429.666667
2,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64477,100.63781,2024-11-01 08:36:41,0,109,1,1,8,4,0,0,0,42429.666667
3,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64477,100.63781,2024-11-01 08:38:42,0,109,1,1,8,4,0,0,0,42429.666667
4,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64477,100.63781,2024-11-01 08:40:41,0,109,1,1,8,4,0,0,0,42429.666667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16141363,zzLYPcDONaA8lLF2aJYFKnoRDQ4,1,13.92919,100.49025,2024-11-30 21:32:47,42,45,1,1,21,5,1,0,0,42573.866667
16141364,zzLYPcDONaA8lLF2aJYFKnoRDQ4,1,13.93869,100.49862,2024-11-30 21:34:47,32,35,1,1,21,5,1,0,0,42573.866667
16141365,zzLYPcDONaA8lLF2aJYFKnoRDQ4,1,13.94617,100.49917,2024-11-30 21:36:47,37,315,1,1,21,5,1,0,0,42573.866667
16141366,zzLYPcDONaA8lLF2aJYFKnoRDQ4,1,13.94629,100.49662,2024-11-30 21:38:47,0,297,1,1,21,5,1,0,0,42573.866667


In [30]:
taxis_df.groupby('trip_id')["VehicleID"].nunique()

trip_id
0      2013
1      1492
2      1464
3      1441
4      1418
       ... 
903       1
904       1
905       1
906       1
907       1
Name: VehicleID, Length: 908, dtype: int64

Filter to Just BMR Bangkok

In [31]:
#Define Regions
BKK_REGION_BOUNDS = {
    'min_lat': 13.4,
    'max_lat': 14.2,
    'min_lon': 99.9,
    'max_lon': 101.3
}

In [32]:
taxis_df_bkk = taxis_df[
    (taxis_df['lat'] >= BKK_REGION_BOUNDS['min_lat']) &
    (taxis_df['lat'] <= BKK_REGION_BOUNDS['max_lat']) &
    (taxis_df['lon'] >= BKK_REGION_BOUNDS['min_lon']) &
    (taxis_df['lon'] <= BKK_REGION_BOUNDS['max_lon'])
].copy()

In [33]:
taxis_df_bkk

,VehicleID,gpsvalid,lat,lon,timestamp,speed,heading,for_hire_light,engine_acc,hour,day_of_week,is_weekend,trip_id,idle_period_id,idle_duration_minutes
0,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64477,100.63777,2024-11-01 08:32:41,0,109,1,1,8,4,0,0,0,42429.666667
1,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64477,100.63777,2024-11-01 08:34:41,0,109,1,1,8,4,0,0,0,42429.666667
2,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64477,100.63781,2024-11-01 08:36:41,0,109,1,1,8,4,0,0,0,42429.666667
3,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64477,100.63781,2024-11-01 08:38:42,0,109,1,1,8,4,0,0,0,42429.666667
4,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64477,100.63781,2024-11-01 08:40:41,0,109,1,1,8,4,0,0,0,42429.666667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16141363,zzLYPcDONaA8lLF2aJYFKnoRDQ4,1,13.92919,100.49025,2024-11-30 21:32:47,42,45,1,1,21,5,1,0,0,42573.866667
16141364,zzLYPcDONaA8lLF2aJYFKnoRDQ4,1,13.93869,100.49862,2024-11-30 21:34:47,32,35,1,1,21,5,1,0,0,42573.866667
16141365,zzLYPcDONaA8lLF2aJYFKnoRDQ4,1,13.94617,100.49917,2024-11-30 21:36:47,37,315,1,1,21,5,1,0,0,42573.866667
16141366,zzLYPcDONaA8lLF2aJYFKnoRDQ4,1,13.94629,100.49662,2024-11-30 21:38:47,0,297,1,1,21,5,1,0,0,42573.866667


In [34]:
#filter speed
taxis_df_bkk = taxis_df_bkk[taxis_df_bkk['speed'] <= 180].copy()

In [35]:
taxis_df_bkk

,VehicleID,gpsvalid,lat,lon,timestamp,speed,heading,for_hire_light,engine_acc,hour,day_of_week,is_weekend,trip_id,idle_period_id,idle_duration_minutes
0,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64477,100.63777,2024-11-01 08:32:41,0,109,1,1,8,4,0,0,0,42429.666667
1,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64477,100.63777,2024-11-01 08:34:41,0,109,1,1,8,4,0,0,0,42429.666667
2,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64477,100.63781,2024-11-01 08:36:41,0,109,1,1,8,4,0,0,0,42429.666667
3,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64477,100.63781,2024-11-01 08:38:42,0,109,1,1,8,4,0,0,0,42429.666667
4,++qQzutWwL31NcUo8s0jiGZzzS0,1,13.64477,100.63781,2024-11-01 08:40:41,0,109,1,1,8,4,0,0,0,42429.666667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16141363,zzLYPcDONaA8lLF2aJYFKnoRDQ4,1,13.92919,100.49025,2024-11-30 21:32:47,42,45,1,1,21,5,1,0,0,42573.866667
16141364,zzLYPcDONaA8lLF2aJYFKnoRDQ4,1,13.93869,100.49862,2024-11-30 21:34:47,32,35,1,1,21,5,1,0,0,42573.866667
16141365,zzLYPcDONaA8lLF2aJYFKnoRDQ4,1,13.94617,100.49917,2024-11-30 21:36:47,37,315,1,1,21,5,1,0,0,42573.866667
16141366,zzLYPcDONaA8lLF2aJYFKnoRDQ4,1,13.94629,100.49662,2024-11-30 21:38:47,0,297,1,1,21,5,1,0,0,42573.866667


In [36]:
#how many taxis extracted
taxis_df_bkk['VehicleID'].nunique()

1934

In [37]:
taxis_df_bkk.groupby('VehicleID')['trip_id'].max().sum()

np.int64(418401)

In [38]:
taxis_df_bkk.columns

Index(['VehicleID', 'gpsvalid', 'lat', 'lon', 'timestamp', 'speed', 'heading',
       'for_hire_light', 'engine_acc', 'hour', 'day_of_week', 'is_weekend',
       'trip_id', 'idle_period_id', 'idle_duration_minutes'],
      dtype='object')

In [39]:
# Save as CSV
taxis_df_bkk.to_csv("cleaned_taxis_11_2024.csv", index=False)

In [40]:
# Example: read data and compute fare
df = pd.read_csv('cleaned_taxis_11_2024.csv')

# Ensure timestamp is in datetime format and sort the data
df['timestamp'] = pd.to_datetime(df['timestamp'])
# We must sort by VehicleID first, then trip_id, then timestamp
df = df.sort_values(by=['VehicleID', 'trip_id', 'timestamp'])


In [41]:
def haversine_distance(lon1, lat1, lon2, lat2):
    """
    Calculate the great circle distance in kilometers between two points
    on the earth (specified in decimal degrees).
    """
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    r = 6371 # Radius of earth in kilometers.
    return c * r

In [42]:
# --- Part 1: Calculate distance between each point (same as before) ---
df['lat_next'] = df.groupby(['VehicleID', 'trip_id'])['lat'].shift(-1)
df['lon_next'] = df.groupby(['VehicleID', 'trip_id'])['lon'].shift(-1)

# This column is the distance between one point and the next, not the total
df['segment_distance_km'] = haversine_distance(df['lon'], df['lat'], df['lon_next'], df['lat_next'])

# --- Part 2: Calculate the TOTAL distance for the whole trip (same as before) ---
# This creates a summary table with one row per trip and its total distance
trip_distances = df.groupby(['VehicleID', 'trip_id'])['segment_distance_km'].sum().reset_index()
trip_distances.rename(columns={'segment_distance_km': 'total_trip_distance_km'}, inplace=True)


# --- Part 3: Merge the total trip distance back to the main dataframe ---
# This is the new step you were looking for.
df = pd.merge(df, trip_distances, on=['VehicleID', 'trip_id'], how='left')

#drop all rows that has trip distance that more than 300 km
df = df[df['total_trip_distance_km'] <= 300]

#drop all rows that has trip distance that equal to 0 km
df = df[df['total_trip_distance_km'] > 0]

In [43]:
def calculate_total_trip_fees(distance_km):
    """
    Calculates the taxi fare in THB based on the provided distance in kilometers,
    using the official Thai taxi rate structure.
    """
    # Handle cases with no or very small distance
    if distance_km <= 0:
        return 0

    # First 1 km
    fare = 35.0

    if distance_km > 1:
        # km >1 to 10
        d = min(distance_km, 10) - 1
        fare += d * 6.50

    if distance_km > 10:
        # km >10 to 20
        d = min(distance_km, 20) - 10
        fare += d * 7.00

    if distance_km > 20:
        # km >20 to 40
        d = min(distance_km, 40) - 20
        fare += d * 8.00

    if distance_km > 40:
        # km >40 to 60
        d = min(distance_km, 60) - 40
        fare += d * 8.50

    if distance_km > 60:
        # km >60 to 80
        d = min(distance_km, 80) - 60
        fare += d * 9.00

    if distance_km > 80:
        # km >80
        d = distance_km - 80
        fare += d * 10.50

    return fare

In [44]:
# (Assuming you have already defined the calculate_total_trip_fees function)
df['total_trip_fees'] = df['total_trip_distance_km'].apply(calculate_total_trip_fees)

In [45]:
# getting all important features
# --- We will group by each unique trip to calculate the new features ---
trip_groups = df.groupby(['VehicleID', 'trip_id'])

# --- 1. Calculate Trip Duration ---
# Calculate duration only for occupied periods
occupied_df = df[df['for_hire_light'] == 0].copy()
trip_durations = occupied_df.groupby(['VehicleID', 'trip_id']).agg({
    'timestamp': ['min', 'max']
}).reset_index()
trip_durations.columns = ['VehicleID', 'trip_id', 'start_time', 'end_time']
trip_durations['duration_minutes'] = (trip_durations['end_time'] - trip_durations['start_time']).dt.total_seconds() / 60

# Merge back to main df
df = pd.merge(df, trip_durations[['VehicleID', 'trip_id', 'duration_minutes']], on=['VehicleID', 'trip_id'], how='left')

In [46]:
df
df = df.dropna()

In [47]:
df

,VehicleID,gpsvalid,lat,lon,timestamp,speed,heading,for_hire_light,engine_acc,hour,...,is_weekend,trip_id,idle_period_id,idle_duration_minutes,lat_next,lon_next,segment_distance_km,total_trip_distance_km,total_trip_fees,duration_minutes
0,+20prWr63K5svsMtTmdqLnmsTGE,1,13.66732,100.58770,2024-11-02 05:10:54,0,292,0,1,5,...,1,0,0,29.0,13.66732,100.58770,0.000000,38.252151,309.517208,17.00
1,+20prWr63K5svsMtTmdqLnmsTGE,1,13.66732,100.58770,2024-11-02 05:12:54,0,292,0,1,5,...,1,0,0,29.0,13.66573,100.59171,0.467952,38.252151,309.517208,17.00
2,+20prWr63K5svsMtTmdqLnmsTGE,1,13.66573,100.59171,2024-11-02 05:14:54,9,112,0,1,5,...,1,0,0,29.0,13.65549,100.59292,1.146117,38.252151,309.517208,17.00
3,+20prWr63K5svsMtTmdqLnmsTGE,1,13.65549,100.59292,2024-11-02 05:16:54,35,173,0,1,5,...,1,0,0,29.0,13.64617,100.59267,1.036689,38.252151,309.517208,17.00
4,+20prWr63K5svsMtTmdqLnmsTGE,1,13.64617,100.59267,2024-11-02 05:18:54,30,195,0,1,5,...,1,0,0,29.0,13.64303,100.59422,0.387246,38.252151,309.517208,17.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11465676,zwNAI8pHuCgPOF5NMs2nVaXBG04,1,13.74413,100.39530,2024-11-29 13:57:07,0,359,1,1,13,...,0,297,297,26.7,13.75599,100.39509,1.318967,13.217055,116.019387,1976.35
11465677,zwNAI8pHuCgPOF5NMs2nVaXBG04,1,13.75599,100.39509,2024-11-29 13:59:08,29,359,1,1,13,...,0,297,297,26.7,13.76797,100.39488,1.332308,13.217055,116.019387,1976.35
11465678,zwNAI8pHuCgPOF5NMs2nVaXBG04,1,13.76797,100.39488,2024-11-29 14:01:07,42,359,1,1,14,...,0,297,297,26.7,13.77637,100.39469,0.934263,13.217055,116.019387,1976.35
11465679,zwNAI8pHuCgPOF5NMs2nVaXBG04,1,13.77637,100.39469,2024-11-29 14:03:07,0,359,1,1,14,...,0,297,297,26.7,13.77977,100.39418,0.382054,13.217055,116.019387,1976.35


In [48]:
df.columns

Index(['VehicleID', 'gpsvalid', 'lat', 'lon', 'timestamp', 'speed', 'heading',
       'for_hire_light', 'engine_acc', 'hour', 'day_of_week', 'is_weekend',
       'trip_id', 'idle_period_id', 'idle_duration_minutes', 'lat_next',
       'lon_next', 'segment_distance_km', 'total_trip_distance_km',
       'total_trip_fees', 'duration_minutes'],
      dtype='object')

In [49]:
trip_groups = df.groupby(['VehicleID', 'trip_id'])
df['pickup_hour'] = trip_groups['hour'].transform('first')
df['pickup_dayofweek'] = trip_groups['day_of_week'].transform('first')
df['average_speed'] = trip_groups['speed'].transform('mean')
df['start_lat'] = trip_groups['lat'].transform('first')
df['start_lon'] = trip_groups['lon'].transform('first')
df['end_lat'] = trip_groups['lat'].transform('last')
df['end_lon'] = trip_groups['lon'].transform('last')

Feature Engineering


In [50]:
# Advanced Feature Engineering for Trip Duration Prediction


# 1. Encode start and end locations into H3 zones (resolution 7)
df['start_h3_zone'] = df.apply(lambda row: h3.latlng_to_cell(row['start_lat'], row['start_lon'], 7), axis=1)
df['end_h3_zone'] = df.apply(lambda row: h3.latlng_to_cell(row['end_lat'], row['end_lon'], 7), axis=1)



# 5. Calculate trip start and end hour difference
# (useful for trips crossing rush hours)
df["is_rush_hour"] = ((df["pickup_hour"].between(7, 9)) | 
                     (df["pickup_hour"].between(17, 19))).astype(int)

# 8. Calculate zone change indicator (1 if start and end zone are different)
df['zone_change'] = (df['start_h3_zone'] != df['end_h3_zone']).astype(int)

# 9. Remove any remaining NaNs (if any)
df = df.dropna()

# 10. Show engineered features
df.head()

# Remove trips with unrealistic durations
df = df[df['duration_minutes'].between(1, 300)]  # 1 min to 5 hours
# Remove trips with zero distance
df = df[df['total_trip_distance_km'] > 0.1]  # At least 100 meters

# ADD THIS BLOCK - convert to trip-level data
trip_level_df = df.groupby(['VehicleID', 'trip_id']).agg({
    'start_lat': 'first',
    'start_lon': 'first', 
    'end_lat': 'first',
    'end_lon': 'first',
    'pickup_hour': 'first',
    'pickup_dayofweek': 'first',
    'total_trip_distance_km': 'first',
    'total_trip_fees': 'first',
    'duration_minutes': 'first',
    'start_h3_zone': 'first',
    'end_h3_zone': 'first',
    'zone_change': 'first',
    'is_rush_hour': 'first',
    'average_speed': 'first'  # This was calculated but not included

}).reset_index()

print(f"Trips before filtering: {len(trip_level_df)}")
trip_level_df = trip_level_df.dropna()
print(f"Trips after removing NaN: {len(trip_level_df)}")
print(f"Duration stats: min={trip_level_df['duration_minutes'].min()}, max={trip_level_df['duration_minutes'].max()}")

# Add straight-line distance
trip_level_df['straight_line_distance'] = trip_level_df.apply(
    lambda row: haversine_distance(row['start_lon'], row['start_lat'], 
                                 row['end_lon'], row['end_lat']), axis=1)

trip_level_df['distance_ratio'] = trip_level_df['total_trip_distance_km'] / (trip_level_df['straight_line_distance'] + 0.001)
trip_level_df['is_weekend'] = (trip_level_df['pickup_dayofweek'] >= 5).astype(int)
trip_level_df['hour_sin'] = np.sin(2 * np.pi * trip_level_df['pickup_hour'] / 24)
trip_level_df['hour_cos'] = np.cos(2 * np.pi * trip_level_df['pickup_hour'] / 24)

Trips before filtering: 352262
Trips after removing NaN: 352262
Duration stats: min=1.0, max=299.9166666666667


In [ ]:
# Save engineered features and targets for analysis
# X_features: all features except the target (duration_minutes)
# y_target: only the target column
from sklearn.preprocessing import LabelEncoder



X_features = trip_level_df.drop(columns=["duration_minutes", "VehicleID", "trip_id"]).copy()
le_start = LabelEncoder()
le_end = LabelEncoder()
X_features['start_h3_zone'] = le_start.fit_transform(X_features['start_h3_zone'])
X_features['end_h3_zone'] = le_end.fit_transform(X_features['end_h3_zone'])


y_target = trip_level_df[['duration_minutes']]

X_features.to_csv('X_features_duration.csv', index=False)
y_target.to_csv('y_target_duration.csv', index=False)

# Quick check
print('Saved X_features_duration.csv and y_target_duration.csv')
X_features.head(), y_target.head()

Saved X_features_duration.csv and y_target_duration.csv


(   start_lat  start_lon   end_lat    end_lon  pickup_hour  pickup_dayofweek  \
 0   13.66732  100.58770  13.91107  100.59643            5                 5   
 1   13.92129  100.60268  13.84257  100.51124            5                 5   
 2   13.84264  100.51028  13.67179  100.45524            6                 5   
 3   13.66222  100.43730  13.65399  100.54064            7                 5   
 4   13.65312  100.53746  13.69322  100.75133            7                 5   
 
    total_trip_distance_km  total_trip_fees  start_h3_zone  end_h3_zone  \
 0               38.252151       309.517208            304          742   
 1               16.787909       141.015361            365          723   
 2               31.257398       253.559186            746          661   
 3               19.787866       162.015063            684          671   
 4               53.763624       440.490808            694          431   
 
    zone_change  is_rush_hour  average_speed  straight_line_distan

: 